# Laser-based 3D reconstruction

This walkthrough explores the **archived point cloud** and runs a **separate synthetic image-pair reconstruction**. The original `colour/` and `laser/` photographs were not supplied. There is no claim that the synthetic example reproduces the archived scan.

In [ ]:
from pathlib import Path
import sys
import numpy as np
from IPython.display import display, Image
ROOT = Path.cwd().resolve()
if not (ROOT / "laser_reconstruction").exists():
    ROOT = ROOT.parent
if not (ROOT / "laser_reconstruction").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory")
sys.path.insert(0, str(ROOT))
from laser_reconstruction.cloud import load_xyz, point_stats
from laser_reconstruction.synthetic import generate_demo
from laser_reconstruction.pipeline import calibration_from_json, reconstruct_sequence
from laser_reconstruction.cloud import write_xyz
print("Project root:", ROOT)

## 1. Examine the original saved scan

These points were read from the `.xyz` file included in the academic archive. Their original laser-on/off source images are unavailable.

In [ ]:
original = load_xyz(ROOT / "data/reference/original_scan.xyz")
point_stats(original)

In [ ]:
display(Image(filename=str(ROOT / "docs/original_point_cloud_preview.png")))

## 2. Produce fictional paired frames

For a fully reproducible example, generate 24 artificial background/laser-on JPEG pairs and a pixel-to-millimeter calibration. These are synthetic software checks, not scanned physical objects.

In [ ]:
demo = ROOT / "outputs/notebook_demo"
calibration_path = generate_demo(demo, frames=24)
H = calibration_from_json(calibration_path)
print("Calibration matrix:\n", H)
print("Generated image pair:", demo / "colour/0000.jpg", demo / "laser/0000.jpg")

## 3. Reconstruct the synthetic turntable sequence

One laser-stripe position per image row is mapped into planar millimeters, rotated by 15 degrees between fictional frames, and assigned an RGB color from the laser-off image.

In [ ]:
cloud, summary = reconstruct_sequence(demo / "colour", demo / "laser", H, frames=24, angle_step_deg=15, strict=True)
summary

In [ ]:
output = ROOT / "outputs/notebook_synthetic.xyz"
write_xyz(output, cloud)
print(point_stats(cloud))
print("Saved to", output)

In [ ]:
display(Image(filename=str(ROOT / "docs/synthetic_point_cloud_preview.png")))

## 4. Interpretation and limits

- The original point cloud is a genuine archived output (300,142 points), but the acquisition image pairs are missing.
- The self-contained example validates the pipeline using artificial image pairs; it cannot establish accuracy on real scans.
- The original calibration file is specific to the unavailable experimental camera setup, and the original report notes CloudCompare orientation correction.

See `docs/technical_notes.md` and the original report for geometry assumptions.